# 📸 gotohp — Thu hồi dung lượng Google Photos (0 Quota)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ntrungz/gotohp/blob/main/colab/gotohp_reclaim.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-ntrungz%2Fgotohp-181717?logo=github)](https://github.com/ntrungz/gotohp)

Chạy lần lượt **4 bước** bên dưới: **Cài đặt** → **Đăng nhập** → **Quét file ăn dung lượng** → **Thu hồi dung lượng**.

> **Lưu ý:** Bản fork từ [`xob0t/gotohp`](https://github.com/xob0t/gotohp) (MIT) & tham khảo API từ [`xob0t/gpmc`](https://github.com/xob0t/gpmc). Lần đầu nên thử `SO_LUONG_TOI_DA = 2` trước khi chạy hàng loạt.

In [ ]:
#@title 1️⃣ Cài đặt môi trường { display-mode: "form" }
#@markdown Bấm **▶** để tự động cài đặt `gotohp-cli`. Bật tùy chọn dưới nếu muốn lưu file backup thẳng vào Google Drive.
LUU_BACKUP_LEN_DRIVE = False #@param {type:"boolean"}

import html
import json
import os
import platform
import shutil
import subprocess
import time
from pathlib import Path
from IPython.display import HTML, display

WORK_DIR = Path("/content/gotohp")
GOTOHP_BIN = Path("/content/gotohp-cli")
PREBUILT_BIN_URL = "https://github.com/ntrungz/gotohp/releases/download/colab-cli/gotohp-cli-linux-amd64"
GO_TARBALL = "go1.27.0.linux-amd64.tar.gz"
GO_BIN_DIR = "/usr/local/go/bin"

if GO_BIN_DIR not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{GO_BIN_DIR}:{os.environ.get('PATH', '')}"


def format_bytes(num_bytes: int) -> str:
    if not num_bytes or num_bytes <= 0:
        return "0 B"
    val = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if val < 1024 or unit == "TB":
            return f"{val:.1f} {unit}"
        val /= 1024
    return f"{val:.1f} TB"


def render_card(title: str, body_html: str, accent: str = "#3b82f6") -> HTML:
    return HTML(
        f"""
        <div style="font-family: 'Inter', system-ui, -apple-system, sans-serif;
                    border: 1px solid #e2e8f0; border-left: 4px solid {accent};
                    border-radius: 12px; padding: 16px 18px;
                    background: #ffffff; color: #334155; margin: 8px 0;
                    box-shadow: 0 1px 2px rgba(15, 23, 42, 0.05);">
          <div style="font-weight: 600; font-size: 15px; margin-bottom: 8px; color: #0f172a;">{title}</div>
          <div style="font-size: 13.5px; line-height: 1.6;">{body_html}</div>
        </div>
        """
    )


def render_kpi_grid(cards: list[tuple[str, str, str, str]]) -> str:
    items = []
    for label, value, sub, color in cards:
        items.append(
            f"""
            <div style="flex: 1 1 150px; background: #f8fafc; border: 1px solid #e2e8f0;
                        border-top: 3px solid {color}; border-radius: 10px; padding: 12px 14px;">
              <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.04em; color: #64748b;">{html.escape(label)}</div>
              <div style="font-size: 22px; font-weight: 700; color: #0f172a; margin: 4px 0;">{html.escape(value)}</div>
              <div style="font-size: 11.5px; color: #475569;">{html.escape(sub)}</div>
            </div>
            """
        )
    return f'<div style="display: flex; flex-wrap: wrap; gap: 10px; margin: 8px 0;">{"".join(items)}</div>'


def get_accounts() -> tuple[str, list[str]]:
    proc = subprocess.run([str(GOTOHP_BIN), "creds", "ls"], capture_output=True, text=True)
    active = ""
    all_emails = []
    for raw in proc.stdout.splitlines():
        line = raw.strip()
        if line.startswith("* "):
            email = line[2:].strip()
            active = email
            all_emails.append(email)
        elif "@" in line and not line.startswith("Use "):
            all_emails.append(line)
    return active, all_emails


def active_account_badge() -> str:
    active, _ = get_accounts()
    if not active:
        return '<span style="background:#fee2e2;color:#b91c1c;border:1px solid #fecaca;padding:2px 8px;border-radius:999px;font-size:12px;font-weight:500;">Chưa đăng nhập</span>'
    return f'<span style="background:#dcfce7;color:#166534;border:1px solid #bbf7d0;padding:2px 10px;border-radius:999px;font-size:12px;font-weight:500;">👤 {html.escape(active)}</span>'


box = display(render_card("⚙️ Đang thiết lập môi trường...", "Vui lòng chờ vài giây..."), display_id=True)
logs = []


def step(msg: str, accent: str = "#3b82f6"):
    logs.append(msg)
    box.update(render_card("⚙️ Cài đặt gotohp", "<br>".join(logs), accent=accent))


try:
    LOCAL_WORK_DIR = "/content/gotohp-reclaim"
    DRIVE_BACKUP_DIR = ""
    BACKUP_DIR = LOCAL_WORK_DIR
    os.makedirs(LOCAL_WORK_DIR, exist_ok=True)
    if LUU_BACKUP_LEN_DRIVE:
        step("📁 Đang kết nối Google Drive...")
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive", force_remount=False)
        DRIVE_BACKUP_DIR = "/content/drive/MyDrive/gotohp-reclaim"
        BACKUP_DIR = DRIVE_BACKUP_DIR
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        step(
            f"📁 Xử lý tạm trên SSD Colab (<code>{html.escape(LOCAL_WORK_DIR)}</code>) "
            f"→ lưu gộp sang Drive (<code>{html.escape(DRIVE_BACKUP_DIR)}</code>) sau khi chạy xong."
        )
    else:
        step(f"📁 Thư mục tạm/backup: <code>{html.escape(BACKUP_DIR)}</code>")

    if GOTOHP_BIN.exists():
        step("✅ Đã sẵn sàng — chuyển sang <b>Bước 2️⃣</b>!", accent="#10b981")
    else:
        downloaded = False
        if platform.machine().lower() in ("x86_64", "amd64"):
            step("⬇️ Đang tải bản biên dịch sẵn <code>gotohp-cli</code>...")
            t0 = time.time()
            tmp_bin = GOTOHP_BIN.with_suffix(".tmp")
            dl = subprocess.run(
                ["curl", "-fsSL", "--retry", "2", "-o", str(tmp_bin), PREBUILT_BIN_URL],
                capture_output=True,
                text=True,
            )
            if dl.returncode == 0 and tmp_bin.exists() and tmp_bin.stat().st_size > 1_000_000:
                tmp_bin.chmod(0o755)
                chk = subprocess.run([str(tmp_bin), "--help"], capture_output=True, text=True)
                if chk.returncode == 0:
                    tmp_bin.replace(GOTOHP_BIN)
                    downloaded = True
                    step(f"✅ Tải xong bản biên dịch sẵn ({time.time() - t0:.1f}s) — sẵn sàng chuyển sang <b>Bước 2️⃣</b>!", accent="#10b981")
            if not downloaded and tmp_bin.exists():
                tmp_bin.unlink()

        if not downloaded:
            step("⚠️ Không tải được bản dựng sẵn — chuyển sang tự biên dịch từ mã nguồn...", accent="#f59e0b")
            need_go = True
            if shutil.which("go"):
                ver = subprocess.check_output(["go", "version"], text=True).strip()
                if "go1.27" in ver or "go1.28" in ver:
                    need_go = False
            if need_go:
                step("⬇️ Đang cài Go 1.27...")
                subprocess.run(["wget", "-q", f"https://go.dev/dl/{GO_TARBALL}", "-O", f"/tmp/{GO_TARBALL}"], check=True)
                subprocess.run(["rm", "-rf", "/usr/local/go"], check=True)
                subprocess.run(["tar", "-C", "/usr/local", "-xzf", f"/tmp/{GO_TARBALL}"], check=True)
                subprocess.run(["rm", "-f", f"/tmp/{GO_TARBALL}"], check=True)

            if not WORK_DIR.exists():
                step("📦 Đang tải mã nguồn gotohp...")
                subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ntrungz/gotohp.git", str(WORK_DIR)], check=True)
            else:
                subprocess.run(["git", "-C", str(WORK_DIR), "pull", "--ff-only"], check=False)

            step("🔨 Đang biên dịch gotohp-cli...")
            t0 = time.time()
            subprocess.run(["go", "build", "-tags", "cli", "-o", str(GOTOHP_BIN), "."], cwd=str(WORK_DIR), check=True)
            step(f"✅ Hoàn tất cài đặt ({time.time() - t0:.1f}s) — sẵn sàng chuyển sang <b>Bước 2️⃣</b>!", accent="#10b981")
except Exception as exc:
    step(f"❌ Lỗi: <code>{html.escape(str(exc))}</code>", accent="#ef4444")
    raise

In [ ]:
#@title 2️⃣ Đăng nhập Google Photos (Tự nhận diện Email từ OAuth) { display-mode: "form" }
#@markdown Bấm **▶** để hiện khung nhập token ngay bên dưới. Cell sẽ **tự dừng chờ** bạn dán token và bấm **🔐 Đăng nhập** (hoặc nhấn **Enter**) rồi mới chạy tiếp sang Bước 3.

import html
import subprocess
import ipywidgets as widgets
from IPython.display import HTML, display

if not GOTOHP_BIN.exists():
    raise RuntimeError("Hãy chạy Bước 1️⃣ trước!")

display(
    render_card(
        "🔑 Lấy nhanh oauth_token (30 giây)",
        '1. Mở <a href="https://accounts.google.com/EmbeddedSetup" target="_blank" style="color:#2563eb;font-weight:500;">accounts.google.com/EmbeddedSetup</a> → đăng nhập → bấm <b>I agree</b> (trang quay vòng là bình thường).<br>'
        '2. Nhấn <code>F12</code> → <b>Application</b> → <b>Cookies</b> → <code>accounts.google.com</code> → copy giá trị <code>oauth_token</code>.<br>'
        '<span style="color:#64748b;font-size:12px;">(Hoặc dán trực tiếp chuỗi <code>androidId=...</code> từ ReVanced.)</span>',
        accent="#8b5cf6",
    )
)


def _prompt_token_ui(attempt: int) -> str:
    try:
        from google.colab import output as colab_output  # type: ignore

        uid = f"gotohp_auth_{attempt}"
        display(
            HTML(
                f"""
                <div id="{uid}_wrap" style="font-family:'Inter',system-ui,-apple-system,sans-serif;
                            background:#ffffff;border:1px solid #e2e8f0;border-left:4px solid #2563eb;
                            border-radius:12px;padding:14px 16px;margin:8px 0;
                            box-shadow:0 1px 2px rgba(15,23,42,0.05);">
                  <div style="font-size:13.5px;font-weight:600;color:#0f172a;margin-bottom:8px;">
                    🔐 Dán <code>oauth_token</code> hoặc <code>androidId=...</code> để đăng nhập
                  </div>
                  <div style="display:flex;flex-wrap:wrap;gap:8px;align-items:center;">
                    <input id="{uid}_input" type="text" autocomplete="off"
                           placeholder="Dán oauth_token hoặc androidId=... rồi bấm Đăng nhập (hoặc Enter)"
                           style="flex:1 1 360px;min-width:260px;padding:9px 12px;font-size:13.5px;
                                  font-family:monospace;color:#0f172a;background:#f8fafc;
                                  border:1px solid #cbd5e1;border-radius:8px;outline:none;" />
                    <button id="{uid}_submit" type="button"
                            style="padding:9px 16px;font-size:13.5px;font-weight:600;color:#ffffff;
                                   background:#2563eb;border:none;border-radius:8px;cursor:pointer;">
                      🔐 Đăng nhập
                    </button>
                  </div>
                </div>
                """
            )
        )
        js = f"""
        new Promise((resolve) => {{
          const inp = document.getElementById('{uid}_input');
          const btn = document.getElementById('{uid}_submit');
          const finish = (val) => {{
            inp.disabled = true;
            btn.disabled = true;
            btn.textContent = '⏳ Đang xử lý...';
            resolve(val);
          }};
          btn.addEventListener('click', () => finish(inp.value.trim()));
          inp.addEventListener('keydown', (e) => {{
            if (e.key === 'Enter') {{
              e.preventDefault();
              finish(inp.value.trim());
            }}
          }});
          inp.focus();
        }})
        """
        return (colab_output.eval_js(js) or "").strip()
    except Exception:
        return input("Dán oauth_token (hoặc androidId=...) rồi bấm Enter: ").strip()


active, emails = get_accounts()
attempt = 0
while not active:
    attempt += 1
    raw = _prompt_token_ui(attempt)
    if not raw:
        display(
            render_card(
                "⚠️ Chưa nhập token",
                "Hãy dán <code>oauth_token</code> (hoặc <code>androidId=...</code>) rồi bấm <b>🔐 Đăng nhập</b>.",
                accent="#f59e0b",
            )
        )
        continue

    proc = subprocess.run(
        [str(GOTOHP_BIN), "creds", "add", "-"],
        input=raw + "\n",
        capture_output=True,
        text=True,
    )
    del raw
    if proc.returncode != 0:
        err = (proc.stderr or proc.stdout).strip() or "Không rõ lỗi"
        display(render_card("❌ Đăng nhập thất bại — thử lại bên dưới", f"<code>{html.escape(err)}</code>", accent="#ef4444"))
        continue

    active, emails = get_accounts()

if len(emails) == 1:
    display(
        render_card(
            "✅ Đã kết nối tài khoản",
            f"{active_account_badge()} — tự động dùng cho các bước tiếp theo.",
            accent="#10b981",
        )
    )
else:
    picker = widgets.Dropdown(
        options=emails,
        value=active if active in emails else emails[0],
        description="Tài khoản:",
        layout=widgets.Layout(width="360px"),
    )
    switch_status = widgets.HTML(value=f"<div style='margin-top:4px;'>{active_account_badge()}</div>")

    def _on_pick(change):
        new_email = change["new"]
        subprocess.run([str(GOTOHP_BIN), "creds", "set", new_email], check=False)
        switch_status.value = (
            f"<div style='margin-top:4px;'>{active_account_badge()} "
            "<span style='color:#64748b;font-size:12px;'>(đã chuyển)</span></div>"
        )

    picker.observe(_on_pick, names="value")
    display(
        render_card(
            "✅ Đã kết nối tài khoản",
            "Chọn tài khoản muốn dùng bên dưới (tự động áp dụng cho Bước 3 & Bước 4).",
            accent="#10b981",
        )
    )
    display(widgets.VBox([picker, switch_status]))

In [ ]:
#@title 3️⃣ Quét file đang ăn dung lượng (Chỉ xem, không xóa) { display-mode: "form" }
#@markdown Bấm **▶** để quét toàn bộ thư viện của tài khoản đang đăng nhập.
SO_FILE_HIEN_THI = 200 #@param {type:"slider", min:10, max:500, step:10}

import html
import json
import subprocess
import threading
import ipywidgets as widgets
from IPython.display import display

if not GOTOHP_BIN.exists():
    raise RuntimeError("Hãy chạy Bước 1️⃣ trước!")
active_email, _ = get_accounts()
if not active_email:
    raise RuntimeError("Hãy đăng nhập ở Bước 2️⃣ trước!")

live_box = display(
    render_card(f"🔍 Đang quét thư viện... {active_account_badge()}", "Đang kết nối Google Photos..."),
    display_id=True,
)

proc = subprocess.Popen(
    [str(GOTOHP_BIN), "reclaim", "--dry-run", "--json-progress"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    bufsize=1,
)
stdout_chunks = []


def _read_stdout():
    assert proc.stdout is not None
    stdout_chunks.append(proc.stdout.read())


t_out = threading.Thread(target=_read_stdout, daemon=True)
t_out.start()

stderr_err = []
assert proc.stderr is not None
for line in proc.stderr:
    line = line.strip()
    if not line:
        continue
    try:
        evt = json.loads(line)
        if evt.get("event") == "scan_progress":
            s = evt.get("data", {})
            live_box.update(
                render_card(
                    f"🔍 Đang quét thư viện... {active_account_badge()}",
                    render_kpi_grid(
                        [
                            ("Trang", str(s.get("pagesScanned", 0)), "Đang quét...", "#0ea5e9"),
                            ("Đã kiểm tra", f"{s.get('totalScanned', 0):,}", "Ảnh & Video", "#6366f1"),
                            ("Ăn dung lượng", f"{s.get('eligibleCount', 0):,}", "Có thể thu hồi", "#f59e0b"),
                            ("Ước tính giải phóng", format_bytes(s.get("totalQuotaBytes", 0)), "Tổng dung lượng", "#10b981"),
                        ]
                    ),
                    accent="#0ea5e9",
                )
            )
    except json.JSONDecodeError:
        stderr_err.append(line)

proc.wait()
t_out.join()

raw_stdout = "".join(stdout_chunks).strip()
if proc.returncode != 0 and not raw_stdout:
    err_msg = "\n".join(stderr_err) or "Quét thất bại"
    live_box.update(render_card("❌ Quét thất bại", f"<pre>{html.escape(err_msg)}</pre>", accent="#ef4444"))
    raise RuntimeError(err_msg)

LAST_SCAN_DATA = json.loads(raw_stdout)
scan_obj = LAST_SCAN_DATA.get("scan") or {}
if scan_obj.get("errorMessage"):
    live_box.update(render_card("❌ Lỗi quét", f"<code>{html.escape(scan_obj['errorMessage'])}</code>", accent="#ef4444"))
    raise RuntimeError(scan_obj["errorMessage"])

with open("/content/gotohp_scan_report.json", "w", encoding="utf-8") as f:
    json.dump(LAST_SCAN_DATA, f, ensure_ascii=False, indent=2)

stats = scan_obj.get("stats") or {}
items = scan_obj.get("items") or []
orig_count = sum(1 for it in items if it.get("isOriginalQuality"))
video_count = sum(1 for it in items if it.get("type") == 2)
skipped = (
    stats.get("skippedTrashed", 0)
    + stats.get("skippedLocked", 0)
    + stats.get("skippedNotOwned", 0)
    + stats.get("skippedNoDedupId", 0)
)

live_box.update(
    render_card(
        f"✅ Quét xong {active_account_badge()}",
        render_kpi_grid(
            [
                ("Tổng đã quét", f"{stats.get('totalScanned', 0):,}", f"{stats.get('pagesScanned', 0)} trang", "#6366f1"),
                ("Ăn dung lượng", f"{len(items):,}", f"Ảnh: {len(items) - video_count:,} · Video: {video_count:,}", "#f59e0b"),
                ("Có thể thu hồi", format_bytes(stats.get("totalQuotaBytes", 0)), f"Gốc: {orig_count:,} · Saver: {len(items) - orig_count:,}", "#10b981"),
                ("Bỏ qua an toàn", f"{skipped:,}", f"Trash: {stats.get('skippedTrashed', 0)} · Locked: {stats.get('skippedLocked', 0)}", "#64748b"),
            ]
        ),
        accent="#10b981",
    )
)

if items:
    search_box = widgets.Text(placeholder="Lọc theo tên file (.mp4, IMG_...)", layout=widgets.Layout(width="280px"))
    kind_box = widgets.Dropdown(
        options=[("Tất cả", "all"), ("Chỉ Original", "orig"), ("Chỉ Saver", "saver"), ("Chỉ Video", "video")],
        value="all",
        layout=widgets.Layout(width="160px"),
    )
    table_html = widgets.HTML()

    def _draw(_=None):
        q = search_box.value.strip().lower()
        k = kind_box.value
        filtered = [
            it
            for it in items
            if (not q or q in (it.get("fileName") or "").lower())
            and (
                k == "all"
                or (k == "orig" and it.get("isOriginalQuality"))
                or (k == "saver" and not it.get("isOriginalQuality"))
                or (k == "video" and it.get("type") == 2)
            )
        ]
        rows = []
        for idx, it in enumerate(filtered[:SO_FILE_HIEN_THI], 1):
            tags = [
                '<span style="background:#dbeafe;color:#1d4ed8;padding:2px 7px;border-radius:999px;font-size:11px;font-weight:500;">Gốc</span>'
                if it.get("isOriginalQuality")
                else '<span style="background:#f1f5f9;color:#475569;padding:2px 7px;border-radius:999px;font-size:11px;font-weight:500;">Saver</span>'
            ]
            if it.get("type") == 2:
                tags.append('<span style="background:#ffedd5;color:#c2410c;padding:2px 7px;border-radius:999px;font-size:11px;font-weight:500;">Video</span>')
            if it.get("isFavorite"):
                tags.append('<span style="background:#fef9c3;color:#a16207;padding:2px 7px;border-radius:999px;font-size:11px;">⭐</span>')
            if it.get("isArchived"):
                tags.append('<span style="background:#e0e7ff;color:#4338ca;padding:2px 7px;border-radius:999px;font-size:11px;">📦</span>')
            rows.append(
                f"<tr style='border-bottom:1px solid #f1f5f9;'>"
                f"<td style='padding:7px 10px;color:#64748b;'>{idx}</td>"
                f"<td style='padding:7px 10px;color:#0f172a;font-family:monospace;max-width:340px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;'>{html.escape(it.get('fileName') or '')}</td>"
                f"<td style='padding:7px 10px;color:#b45309;font-weight:600;text-align:right;white-space:nowrap;'>{format_bytes(it.get('quotaChargedBytes', 0))}</td>"
                f"<td style='padding:7px 10px;'>{' '.join(tags)}</td>"
                f"</tr>"
            )
        rows_body = "".join(rows) or '<tr><td colspan="4" style="padding:16px;text-align:center;color:#64748b;">Không có file nào khớp</td></tr>'
        table_html.value = (
            f"<div style='font-family:system-ui;background:#ffffff;border:1px solid #e2e8f0;border-radius:10px;overflow:hidden;margin-top:6px;box-shadow:0 1px 2px rgba(15,23,42,0.04);'>"
            f"<div style='padding:8px 12px;background:#f8fafc;border-bottom:1px solid #e2e8f0;color:#475569;font-size:12px;'>"
            f"Hiển thị <b>{min(len(filtered), SO_FILE_HIEN_THI)}</b> / <b>{len(filtered):,}</b> file (nặng nhất lên đầu)</div>"
            f"<div style='max-height:320px;overflow-y:auto;'>"
            f"<table style='width:100%;border-collapse:collapse;font-size:12.5px;'><tbody>"
            f"{rows_body}"
            f"</tbody></table></div></div>"
        )

    search_box.observe(_draw, names="value")
    kind_box.observe(_draw, names="value")
    _draw()
    display(widgets.VBox([widgets.HBox([search_box, kind_box]), table_html]))
else:
    display(render_card("🎉 Không có file nào ăn dung lượng!", "Tài khoản của bạn đã sạch dung lượng Google Photos.", accent="#10b981"))

In [ ]:
#@title 4️⃣ Chạy Thu hồi Dung lượng (Download → Trash → Up lại 0 Quota) { display-mode: "form" }
#@markdown Tự động dùng tài khoản đang đăng nhập và thư mục backup ở Bước 1. Khuyên chạy thử `SO_LUONG_TOI_DA = 2` lần đầu.
SO_LUONG_TOI_DA = 200 #@param {type:"integer"}
SO_LUONG_CHAY_SONG_SONG = 4 #@param {type:"slider", min:1, max:10, step:1}
CHAT_LUONG_UP_LAI = "Chất lượng gốc (Pixel XL - 0 quota)" #@param ["Chất lượng gốc (Pixel XL - 0 quota)", "Tiết kiệm dung lượng (Pixel 2 - 0 quota)"]
XOA_VINH_VIEN_KHOI_TRASH = False #@param {type:"boolean"}
XOA_FILE_TAM_SAU_KHI_UP = True #@param {type:"boolean"}
KHOI_PHUC_METADATA = True #@param {type:"boolean"}
XAC_NHAN_XOA_VINH_VIEN = "" #@param {type:"string"}

import html
import json
import os
import shutil
import subprocess
import threading
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

if not GOTOHP_BIN.exists():
    raise RuntimeError("Hãy chạy Bước 1️⃣ trước!")
active_email, _ = get_accounts()
if not active_email:
    raise RuntimeError("Hãy đăng nhập ở Bước 2️⃣ trước!")

if XOA_VINH_VIEN_KHOI_TRASH and XAC_NHAN_XOA_VINH_VIEN.strip().upper() != "YES":
    display(
        render_card(
            "🛑 Cần xác nhận xóa vĩnh viễn",
            "Bạn đang bật <b>XOA_VINH_VIEN_KHOI_TRASH</b>. Hãy gõ <b>YES</b> vào ô <code>XAC_NHAN_XOA_VINH_VIEN</code> rồi bấm ▶ lại.",
            accent="#ef4444",
        )
    )
    raise SystemExit("Cần gõ YES để xác nhận")

local_work_dir = globals().get("LOCAL_WORK_DIR", "/content/gotohp-reclaim")
drive_backup_dir = globals().get("DRIVE_BACKUP_DIR", "")
backup_dir = drive_backup_dir or local_work_dir
os.makedirs(local_work_dir, exist_ok=True)
use_saver = CHAT_LUONG_UP_LAI.startswith("Tiết kiệm")

cmd = [
    str(GOTOHP_BIN),
    "reclaim",
    "--json-progress",
    "--download-dir",
    local_work_dir,
    "--threads",
    str(max(1, int(SO_LUONG_CHAY_SONG_SONG))),
]
if int(SO_LUONG_TOI_DA) > 0:
    cmd += ["--limit", str(int(SO_LUONG_TOI_DA))]
if XOA_VINH_VIEN_KHOI_TRASH:
    cmd += ["--delete-permanently"]
if XOA_FILE_TAM_SAU_KHI_UP:
    cmd += ["--cleanup-local"]
if use_saver:
    cmd += ["--saver"]
if not KHOI_PHUC_METADATA:
    cmd += ["--no-restore-metadata"]

prog_bar = widgets.IntProgress(value=0, min=0, max=1, bar_style="info", layout=widgets.Layout(width="100%"))
live_card = widgets.HTML()
live_log = widgets.HTML()
display(widgets.VBox([prog_bar, live_card, live_log]))

STAGES = {
    "downloading": "⬇️ Tải bản gốc",
    "trashing": "🗑️ Đưa vào Trash",
    "deleting": "🔥 Xóa khỏi Trash",
    "uploading": "⬆️ Up lại (0 quota)",
    "restoring_metadata": "🏷️ Khôi phục metadata",
    "restoring_trash": "↩️ Khôi phục từ Trash",
    "cleaning_up": "🧹 Dọn file tạm",
    "completed": "✅ Xong",
    "error": "❌ Lỗi",
}

st = {"total": 0, "total_bytes": 0, "done": 0, "ok": 0, "fail": 0, "reclaimed": 0, "workers": {}, "recent": []}


def _render_live(header_override: str = ""):
    header = header_override or (
        f"{active_account_badge()} &nbsp; "
        f"Tiến độ: <b>{st['done']}/{st['total']}</b> · "
        f"✅ <b style='color:#15803d;'>{st['ok']}</b> · "
        f"❌ <b style='color:#dc2626;'>{st['fail']}</b> · "
        f"💾 <b style='color:#b45309;'>{format_bytes(st['reclaimed'])}</b> / {format_bytes(st['total_bytes'])}"
    )
    w_rows = []
    for wid in sorted(st["workers"].keys()):
        w = st["workers"][wid]
        label = STAGES.get(w.get("stage", ""), w.get("stage", ""))
        bd, bt = w.get("bytesDone", 0), w.get("bytesTotal", 0)
        pct = f" · {min(100, int(bd * 100 / bt))}%" if bt > 0 else ""
        w_rows.append(
            f"<div style='padding:5px 10px;margin-top:6px;background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px;font-size:12.5px;color:#0f172a;display:flex;justify-content:space-between;gap:8px;'>"
            f"<span style='overflow:hidden;text-overflow:ellipsis;white-space:nowrap;font-family:monospace;'>{html.escape(w.get('fileName', ''))}</span>"
            f"<span style='color:#2563eb;font-weight:500;white-space:nowrap;'>{label}{pct}</span></div>"
        )
    live_card.value = (
        f"<div style='font-family:system-ui;background:#ffffff;border:1px solid #e2e8f0;border-radius:10px;padding:12px 14px;color:#1e293b;margin-top:4px;box-shadow:0 1px 2px rgba(15,23,42,0.04);'>"
        f"<div style='font-size:13px;'>{header}</div>{''.join(w_rows)}</div>"
    )
    if st["recent"]:
        rows = []
        for r in reversed(st["recent"][-10:]):
            icon = "❌" if r.get("isError") else "✅"
            msg = html.escape(r.get("errorMessage") or f"Thu hồi {format_bytes(r.get('quotaChargedBytes', 0))}")
            if r.get("restoredFromTrash"):
                msg += " (↩️ đã khôi phục từ Trash)"
            rows.append(
                f"<div style='padding:5px 10px;border-bottom:1px solid #e2e8f0;font-size:12px;'>"
                f"{icon} <b>{html.escape(r.get('fileName', ''))}</b> — <span style='color:#475569;'>{msg}</span></div>"
            )
        live_log.value = (
            f"<div style='font-family:system-ui;background:#f8fafc;border:1px solid #e2e8f0;border-radius:10px;margin-top:6px;max-height:190px;overflow-y:auto;color:#1e293b;'>"
            f"{''.join(rows)}</div>"
        )


_render_live(f"{active_account_badge()} &nbsp; 🔍 Đang quét thư viện...")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1)
stdout_chunks = []


def _read_stdout():
    assert proc.stdout is not None
    stdout_chunks.append(proc.stdout.read())


t_out = threading.Thread(target=_read_stdout, daemon=True)
t_out.start()

stderr_err = []
assert proc.stderr is not None
for line in proc.stderr:
    line = line.strip()
    if not line:
        continue
    try:
        evt = json.loads(line)
    except json.JSONDecodeError:
        stderr_err.append(line)
        continue
    et, d = evt.get("event"), evt.get("data") or {}
    if et == "scan_progress":
        _render_live(
            f"{active_account_badge()} &nbsp; 🔍 Đang quét trang <b>{d.get('pagesScanned', 0)}</b> · "
            f"Tìm thấy <b>{d.get('eligibleCount', 0):,}</b> file ({format_bytes(d.get('totalQuotaBytes', 0))})"
        )
    elif et == "reclaim_start":
        st["total"] = d.get("total", 0)
        st["total_bytes"] = d.get("totalQuotaBytes", 0)
        prog_bar.max = max(1, st["total"])
        prog_bar.value = 0
        _render_live()
    elif et == "item_progress":
        st["workers"][d.get("workerId", 0)] = d
        _render_live()
    elif et == "item_result":
        st["done"] += 1
        if d.get("isError"):
            st["fail"] += 1
        else:
            st["ok"] += 1
            st["reclaimed"] += d.get("quotaChargedBytes", 0)
        st["recent"].append(d)
        prog_bar.value = min(prog_bar.max, st["done"])
        _render_live()

try:
    proc.wait()
    t_out.join()
finally:
    if drive_backup_dir:
        src_root = Path(local_work_dir)
        dst_root = Path(drive_backup_dir)
        remaining_files = [p for p in src_root.rglob("*") if p.is_file()]
        if remaining_files:
            sync_box = display(
                render_card(
                    "☁️ Đang lưu gộp file backup lên Google Drive...",
                    f"Đang chuyển <b>{len(remaining_files):,}</b> file sang <code>{html.escape(drive_backup_dir)}</code>...",
                    accent="#2563eb",
                ),
                display_id=True,
            )
            moved_count = 0
            moved_bytes = 0
            for src_file in remaining_files:
                rel = src_file.relative_to(src_root)
                dst_file = dst_root / rel
                dst_file.parent.mkdir(parents=True, exist_ok=True)
                fsize = src_file.stat().st_size
                shutil.copy2(src_file, dst_file)
                src_file.unlink()
                moved_count += 1
                moved_bytes += fsize
            for d in sorted([p for p in src_root.rglob("*") if p.is_dir()], reverse=True):
                try:
                    d.rmdir()
                except OSError:
                    pass
            sync_box.update(
                render_card(
                    "☁️ Đã lưu backup lên Google Drive",
                    f"Đã chuyển gộp <b>{moved_count:,}</b> file ({format_bytes(moved_bytes)}) sang <code>{html.escape(drive_backup_dir)}</code>.",
                    accent="#10b981",
                )
            )

raw_stdout = "".join(stdout_chunks).strip()
if not raw_stdout:
    prog_bar.bar_style = "danger"
    err = "\n".join(stderr_err) or "Không có dữ liệu trả về"
    display(render_card("❌ Lỗi chạy Reclaim", f"<pre>{html.escape(err)}</pre>", accent="#ef4444"))
    raise RuntimeError(err)

LAST_RECLAIM_OUTPUT = json.loads(raw_stdout)
with open("/content/gotohp_reclaim_result.json", "w", encoding="utf-8") as f:
    json.dump(LAST_RECLAIM_OUTPUT, f, ensure_ascii=False, indent=2)

summary = LAST_RECLAIM_OUTPUT.get("summary") or {}
has_err = bool(summary.get("errorMessage")) or summary.get("failed", 0) > 0
prog_bar.value = prog_bar.max
prog_bar.bar_style = "warning" if has_err else "success"

body = render_kpi_grid(
    [
        ("Đã xử lý", str(summary.get("total", 0)), "Tổng mục", "#6366f1"),
        ("Thành công", str(summary.get("succeeded", 0)), "Đã up lại 0 quota", "#10b981"),
        ("Thất bại", str(summary.get("failed", 0)), "Giữ backup an toàn", "#ef4444" if summary.get("failed", 0) else "#64748b"),
        ("Đã thu hồi", format_bytes(summary.get("reclaimedBytes", 0)), f"Backup: {backup_dir}", "#f59e0b"),
    ]
)
if summary.get("errorMessage"):
    body += f"<div style='color:#b91c1c;margin-top:6px;'>⚠️ {html.escape(summary['errorMessage'])}</div>"

display(render_card("⚠️ Hoàn tất (có mục lỗi)" if has_err else "🎉 Thu hồi dung lượng thành công!", body, accent="#f59e0b" if has_err else "#10b981"))

In [ ]:
#@title 5️⃣ Tải Backup về máy & Dọn dẹp phiên Colab { display-mode: "form" }
#@markdown Bấm **▶** để hiện các nút thao tác nhanh (không cần sửa code).

import html
import os
import shutil
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

backup_dir = Path(globals().get("BACKUP_DIR", "/content/gotohp-reclaim"))
out_area = widgets.Output()

btn_zip = widgets.Button(description="📦 Tải ZIP Backup", button_style="primary", layout=widgets.Layout(width="180px"))
btn_json = widgets.Button(description="📄 Tải Báo cáo JSON", button_style="info", layout=widgets.Layout(width="180px"))
btn_wipe = widgets.Button(description="🧹 Xóa Token khỏi Colab", button_style="danger", layout=widgets.Layout(width="200px"))


def _download_zip(_):
    with out_area:
        out_area.clear_output()
        if not backup_dir.exists() or not any(backup_dir.iterdir()):
            display(render_card("ℹ️ Chưa có file backup", f"Thư mục <code>{html.escape(str(backup_dir))}</code> đang trống.", accent="#f59e0b"))
            return
        from google.colab import files  # type: ignore

        zip_path = shutil.make_archive("/content/gotohp_reclaim_backups", "zip", str(backup_dir))
        display(render_card("📦 Đang tải ZIP Backup...", f"<code>{html.escape(zip_path)}</code> ({format_bytes(os.path.getsize(zip_path))})", accent="#10b981"))
        files.download(zip_path)


def _download_json(_):
    with out_area:
        out_area.clear_output()
        from google.colab import files  # type: ignore

        found = [p for p in ["/content/gotohp_scan_report.json", "/content/gotohp_reclaim_result.json"] if os.path.exists(p)]
        if not found:
            display(render_card("ℹ️ Chưa có báo cáo", "Hãy chạy Bước 3 hoặc Bước 4 trước.", accent="#f59e0b"))
            return
        for p in found:
            files.download(p)
        display(render_card("📄 Đã tải báo cáo JSON", "<br>".join(f"• <code>{html.escape(p)}</code>" for p in found), accent="#10b981"))


def _wipe_creds(_):
    with out_area:
        out_area.clear_output()
        removed = []
        for p in [Path.home() / ".config" / "gotohp", Path("/content/gotohp.config"), Path("/content/gotohp/gotohp.config")]:
            if p.exists():
                shutil.rmtree(p) if p.is_dir() else p.unlink()
                removed.append(str(p))
        if removed:
            display(render_card("🧹 Đã xóa sạch credential khỏi Colab", "<br>".join(f"• <code>{html.escape(p)}</code>" for p in removed), accent="#10b981"))
        else:
            display(render_card("✅ Phiên Colab đã sạch", "Không còn file credential nào trên máy ảo.", accent="#10b981"))


btn_zip.on_click(_download_zip)
btn_json.on_click(_download_json)
btn_wipe.on_click(_wipe_creds)
display(widgets.VBox([widgets.HBox([btn_zip, btn_json, btn_wipe]), out_area]))